# L05 – MLP och aktiveringsfunktioner

**Tid:** ca 2 h · **Del B: Träna ett nätverk väl**

## Mål
1. Förstå **varför** ett nätverk behöver icke-linjära aktiveringsfunktioner.
2. Välja aktiveringsfunktion: varför **ReLU** är standard och sigmoid sällan används i dolda lager.
3. Bygga och träna en **MLP** (multilayer perceptron) och räkna dess parametrar.
4. Samla träningskoden i ett eget litet paket, **`dlkit`**, som används i resten av kursen.

**Var i arbetsflödet?** Steg 5–6: modell och träning. Baslinjen från L04 (linjär modell, cirka 83 %) är det vi ska slå.

In [ ]:
import random
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

def set_seed(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = "../data"
print("device:", device)

Samma datapipeline som i L04. Värdena för `MEAN` och `STD` beräknade vi där, på enbart train.

In [ ]:
MEAN, STD = 0.2857, 0.3529
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))])
full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=transform)
train_ds, val_ds = random_split(full_train, [50_000, 10_000], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=2)

## 1. Varför behövs icke-linjäritet?

En idé är att få en kraftfullare modell genom att lägga flera `Linear`-lager efter varandra. Men två linjära lager efter varandra är **fortfarande linjärt**:

$$W_2(W_1 x + b_1) + b_2 = \underbrace{(W_2 W_1)}_{W} x + \underbrace{(W_2 b_1 + b_2)}_{b}$$

Hur många lager du än staplar kan nätet bara rita *raka* beslutsgränser, precis som logistisk regression. Vi kontrollerar det numeriskt:

In [ ]:
set_seed(0)
lin1, lin2 = nn.Linear(4, 8), nn.Linear(8, 3)
x = torch.randn(5, 4)

W = lin2.weight @ lin1.weight                    # (3, 8) @ (8, 4) = (3, 4)
b = lin2.weight @ lin1.bias + lin2.bias
print("två lager == ett lager:", torch.allclose(lin2(lin1(x)), x @ W.T + b, atol=1e-6))

Lösningen är en **aktiveringsfunktion**, en enkel icke-linjär funktion, mellan lagren. Med den kan nätverket böja och vika ytan och lära sig godtyckligt komplicerade samband. Det är vad ett neuralt nät tillför jämfört med linjära modeller.

## 2. Aktiveringsfunktioner

| Funktion | Formel | Utdata | Kommentar |
|---|---|---|---|
| **Sigmoid** | $1/(1+e^{-x})$ | (0, 1) | Klassisk. Derivatan är högst 0,25 och nära 0 för stora \|x\| (*mättnad*). |
| **Tanh** | $\tanh(x)$ | (−1, 1) | Centrerad kring 0, men mättas också. |
| **ReLU** | $\max(0, x)$ | [0, ∞) | Derivatan är exakt 1 för positiva x. Snabb och enkel. **Standardvalet.** |
| **GELU** | $x \cdot \Phi(x)$ | ≈ [−0,17, ∞) | En mjuk variant av ReLU. Används i transformers (L13). |

In [ ]:
x = torch.linspace(-5, 5, 200, requires_grad=True)
acts = {"sigmoid": torch.sigmoid, "tanh": torch.tanh, "ReLU": torch.relu, "GELU": nn.functional.gelu}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, f in acts.items():
    y = f(x)
    grad, = torch.autograd.grad(y.sum(), x)   # derivatan i varje punkt
    axes[0].plot(x.detach(), y.detach(), label=name)
    axes[1].plot(x.detach(), grad, label=name)
axes[0].set_title("Aktiveringsfunktion"); axes[1].set_title("Derivata")
for ax in axes: ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Varför är derivatan viktig?** Under backward multipliceras derivatorna lager för lager (kedjeregeln). Med sigmoid blir varje faktor högst 0,25. Med 10 lager blir det högst $0{,}25^{10} \approx 10^{-6}$. De tidiga lagren får nästan ingen gradient och slutar lära sig. Det kallas **försvinnande gradienter** (*vanishing gradients*). ReLU har derivatan 1 för positiva värden, så gradienten passerar oförändrad.

**När används sigmoid då?** Som *utdata* när man vill ha en sannolikhet för binär klassificering. Det sköts i så fall av `BCEWithLogitsLoss`. Den används också inuti LSTM (L12). Den används inte i dolda lager.

## 3. Bygg en MLP

En **MLP** består av linjära lager med aktiveringsfunktioner emellan. Vi bygger den med `nn.Sequential`, där lagren körs i ordning:

In [ ]:
model = nn.Sequential(
    nn.Flatten(),             # (N, 1, 28, 28) → (N, 784): en MLP vill ha vektorer
    nn.Linear(784, 256),      # dolt lager 1
    nn.ReLU(),
    nn.Linear(256, 128),      # dolt lager 2
    nn.ReLU(),
    nn.Linear(128, 10),       # utdatalager: 10 logits, INGEN aktivering (CrossEntropy vill ha logits)
)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f"antal parametrar: {n_params:,}")

**Varför just de här valen?**
- **784 in:** 28 × 28 pixlar. Det bestäms av datan.
- **10 ut:** en logit per klass. Det bestäms av problemet.
- **256 och 128 dolda neuroner:** ett vanligt mönster är att bredden minskar mot utdatan och att storlekarna är potenser av 2. Det är tradition och fungerar bra med GPU:er, men det finns inget magiskt med det. Bredd och djup är **hyperparametrar** som du väljer på valideringsmängden (L08).
- **ReLU** efter varje dolt lager, men **inte** efter det sista.

**Parametrar:** `Linear(784, 256)` har 784 × 256 vikter och 256 bias, totalt 200 960. Hela nätet har cirka 235 000 parametrar. Det är många jämfört med 50 000 träningsbilder, och risken för överanpassning tar vi upp i L06–L07.

## 4. Träna MLP:n

Vi skriver två hjälpfunktioner. De är en förlängning av L04, och nu loggar vi även loss.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    model.train()                                  # träningsläge (betydelse i L07)
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)         # loss är ett medel över batchen → vikta med storleken
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()                                   # utvärderingsläge
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n

In [ ]:
set_seed(42)
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)   # standardvalet från L03

for epoch in range(1, 6):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
    print(f"epok {epoch} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time()-t0:.1f}s")

MLP:n bör nå cirka **88–89 %** val accuracy efter 5 epoker, jämfört med cirka 83 % för den linjära modellen i L04. Icke-linjäriteten gör skillnad.

### Jämförelse med sklearns `MLPClassifier`
sklearn har samma modell. Skillnaderna är att den bara körs på CPU och att du inte kan styra loopen. Vi tränar den på ett urval av 10 000 bilder, eftersom det annars går långsamt:

In [ ]:
from sklearn.neural_network import MLPClassifier

X_sk = ((full_train.data[train_ds.indices[:10_000]].float() / 255 - MEAN) / STD).reshape(10_000, -1).numpy()
y_sk = full_train.targets[train_ds.indices[:10_000]].numpy()
X_sk_val = ((full_train.data[val_ds.indices].float() / 255 - MEAN) / STD).reshape(10_000, -1).numpy()
y_sk_val = full_train.targets[val_ds.indices].numpy()

t0 = time.time()
sk_mlp = MLPClassifier(hidden_layer_sizes=(256, 128), activation="relu", solver="adam",
                       learning_rate_init=1e-3, batch_size=128, max_iter=10, random_state=42)
sk_mlp.fit(X_sk, y_sk)
print(f"sklearn MLP (10k bilder): val accuracy {sk_mlp.score(X_sk_val, y_sk_val):.3f}, {time.time()-t0:.0f}s")

Parametrarna `hidden_layer_sizes`, `activation`, `solver`, `learning_rate_init` och `batch_size` motsvarar exakt de val vi gjorde ovan. Varningen om konvergens betyder att `max_iter` (antalet epoker) tog slut.

## 5. Vad händer om … vi byter aktiveringsfunktion eller storlek?

**Gissa först:** hur går det för sigmoid jämfört med ReLU efter 3 epoker? Och hur stor betydelse har nätets storlek?

In [ ]:
def make_mlp(hidden_sizes, activation=nn.ReLU):
    layers, in_features = [nn.Flatten()], 784
    for h in hidden_sizes:
        layers += [nn.Linear(in_features, h), activation()]
        in_features = h
    layers.append(nn.Linear(in_features, 10))
    return nn.Sequential(*layers)

def quick_run(model, epochs=3, lr=1e-3):
    set_seed(42)
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        train_one_epoch(model, train_loader, loss_fn, opt, device)
    return evaluate(model, val_loader, loss_fn, device)[1]

for name, act in [("ReLU", nn.ReLU), ("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh)]:
    set_seed(42)
    print(f"{name:8s} (3 dolda lager): val acc {quick_run(make_mlp([256, 256, 256], act)):.3f}")

In [ ]:
for sizes in [[32], [256], [256, 128], [512, 512, 512]]:
    set_seed(42)
    m = make_mlp(sizes)
    n = sum(p.numel() for p in m.parameters())
    print(f"dolda lager {str(sizes):16s} {n:>9,} parametrar  val acc {quick_run(m):.3f}")

**Vad vi ser:**
- Med bara några få lager och Adam klarar sig sigmoid och tanh hyggligt, men ReLU är lika bra eller bättre och snabbare. Skillnaden växer med djupet.
- Större nät ger bättre resultat upp till en gräns. Därefter blir vinsten liten medan risken för överanpassning ökar. Redan `[256]` är nära de större näten.

**Tumregel för en MLP:** börja med 1–2 dolda lager med 128–512 neuroner och ReLU. Öka storleken bara om modellen *underanpassar* (L06).

## 6. Flytta koden till `dlkit`

Nu har du skrivit träningsloopen för hand flera gånger (L03, L04 och L05). Från och med nu samlar vi den i ett eget litet paket, ungefär som att gå från lös kod till en sklearn-`Pipeline`. Senare lektioner kan då fokusera på det nya.

Cellerna nedan skriver filerna till `dl-course/dlkit/`. Läs igenom koden: det är samma funktioner som ovan.

In [ ]:
%%writefile ../dlkit/__init__.py
"""dlkit – kursens egna hjälpfunktioner. Byggs upp lektion för lektion."""

In [ ]:
%%writefile ../dlkit/seed.py
import random

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    """Fix all random generators (Python, NumPy, PyTorch CPU/GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
%%writefile ../dlkit/data.py
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms

# dl-course/data, independent of the notebook's working directory
DATA_DIR = Path(__file__).resolve().parent.parent / "data"

# Computed on the 50 000 training images only (L04)
FASHION_MEAN, FASHION_STD = 0.2857, 0.3529
FASHION_CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                   "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


def fashion_mnist_loaders(batch_size=128, train_subset=None, seed=42, num_workers=2):
    """Return (train_loader, val_loader, test_loader) for Fashion-MNIST.

    The 60 000 official training images are split into 50 000 train / 10 000 val.
    train_subset: use only the first n training images (to provoke overfitting).
    """
    tf = transforms.Compose([transforms.ToTensor(),
                             transforms.Normalize((FASHION_MEAN,), (FASHION_STD,))])
    full_train = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=tf)
    test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=tf)
    train_ds, val_ds = random_split(full_train, [50_000, 10_000],
                                    generator=torch.Generator().manual_seed(seed))
    if train_subset is not None:
        train_ds = Subset(train_ds, range(train_subset))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=256, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader

In [ ]:
%%writefile ../dlkit/train.py
import time

import torch


def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """One pass over the training data. Returns (mean loss, accuracy)."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    """Evaluate without gradients. Returns (mean loss, accuracy)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def predict(model, loader, device):
    """Return (y_true, y_pred, probs) as NumPy arrays – ready for sklearn.metrics."""
    model.eval()
    ys, probs = [], []
    for X, y in loader:
        logits = model(X.to(device))
        probs.append(torch.softmax(logits, dim=1).cpu())
        ys.append(y)
    probs = torch.cat(probs)
    return torch.cat(ys).numpy(), probs.argmax(dim=1).numpy(), probs.numpy()


def fit(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=10, verbose=True):
    """Train for a number of epochs. Returns a history dict with loss/accuracy per epoch."""
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
        for key, value in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[key].append(value)
        if verbose:
            print(f"epok {epoch:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.3f} | {time.time() - t0:.1f}s")
    return history

In [ ]:
%%writefile ../dlkit/plots.py
import matplotlib.pyplot as plt


def plot_history(history, title=None):
    """Plot learning curves (loss and accuracy) for train and val."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["loss", "acc"]):
        ax.plot(epochs, history[f"train_{metric}"], "o-", label="train")
        ax.plot(epochs, history[f"val_{metric}"], "o-", label="val")
        ax.set_xlabel("epok")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.legend()
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

Testa att paketet fungerar. `sys.path.insert(0, "..")` gör att Python hittar `dlkit` i mappen ovanför `notebooks/`. Den raden finns överst i alla notebooks från och med L06.

In [ ]:
import sys
sys.path.insert(0, "..")

from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders
from dlkit.train import fit
from dlkit.plots import plot_history

set_seed(42)
train_loader, val_loader, test_loader = fashion_mnist_loaders()
model = make_mlp([256, 128]).to(device)
history = fit(model, train_loader, val_loader, nn.CrossEntropyLoss(),
              torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=5)
plot_history(history, "MLP [256, 128]")

## 7. Övningar

### Övning 1 – räkna parametrar
Bygg en MLP med `nn.Sequential`: Flatten, 784→100, ReLU, 100→50, ReLU, 50→10. Räkna först för hand hur många parametrar den har, och kontrollera sedan.

In [ ]:
my_mlp = None          # TODO
my_count_by_hand = 0   # TODO: din handräkning

In [ ]:
assert sum(p.numel() for p in my_mlp.parameters()) == my_count_by_hand == 84_060
print("Rätt!")

### Övning 2 – slå 88 %
Använd `dlkit` (`fashion_mnist_loaders`, `fit`) och träna en MLP som når **minst 88 % val accuracy**. Du väljer själv arkitektur, learning rate och antal epoker, men varje träning ska ta under cirka 2 minuter. Skriv i en markdown-cell vad du provade och varför.

In [ ]:
set_seed(42)
# TODO
history = None

In [ ]:
assert max(history["val_acc"]) >= 0.88, max(history["val_acc"])
print("Rätt!", max(history["val_acc"]))

### Övning 3 – reflektion
1. Varför har det sista lagret ingen ReLU?
2. Du lägger till 10 dolda lager med sigmoid och träningen står still. Vad är den troliga orsaken?
3. När är en MLP **fel** val? (Tänk på bilder med 1 000 × 1 000 pixlar.)

*Dina svar:*

## När använder jag detta?
- **MLP:** för tabelldata och små, enkla indata. Den är också "huvudet" som klassificerar i större nätverk (CNN, transformers). För tabelldata är dock gradient boosting (till exempel `HistGradientBoostingClassifier`) ofta lika bra eller bättre, och det är en bra baslinje.
- **ReLU** i dolda lager som standard. **GELU** i transformers. **Sigmoid och tanh** bara i utdata eller i speciella strukturer (LSTM).
- **Sista lagret** har ingen aktivering. Lossen tar hand om det.
- **Storlek:** börja litet och öka vid underanpassning.

## Sammanfattning
- Utan aktiveringsfunktion är ett djupt nät bara en linjär modell.
- ReLU är standard eftersom gradienten inte försvinner för positiva värden.
- En MLP med två dolda lager ger cirka 88 % på Fashion-MNIST, jämfört med 83 % för den linjära modellen.
- `dlkit` innehåller nu `set_seed`, `fashion_mnist_loaders`, `train_one_epoch`, `evaluate`, `predict`, `fit` och `plot_history`.

**Nästa lektion:** hur man utvärderar ordentligt, med mått, inlärningskurvor och felanalys.